# 02 — Tissue and TMA core extraction

- `rp.extract_tissue` cuts every separate piece of tissue out of ordinary
  slides (biopsies, resections).
- `rp.extract_tma` cuts round cores out of tissue-microarray blocks, using the
  H&E slide to find cores and cutting the same boxes from each IHC slide.

Both need the `extraction` extra. Add the `semantic` extra and
`detector="semantic"` to use a TIAToolbox tissue model instead of Otsu.

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts at its root or in how_to_use/."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "rocqipath").is_dir():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"          # your slides (kept out of git)
RESULTS_ROOT = PROJECT_ROOT / "results"    # outputs (kept out of git)
DEMO_ROOT = PROJECT_ROOT / "notebook_demo_outputs"  # synthetic examples

import rocqipath as rp

print(f"RocqiPath {rp.__version__}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")

In [ ]:
WSI_INPUT_DIR = DATA_ROOT / "wsi"
TMA_INPUT_DIR = DATA_ROOT / "tma"

TARGET_MAGNIFICATION = 20.0
WSI_SOURCE_MAGNIFICATION = None   # set only when metadata is absent
TMA_SOURCE_MAGNIFICATION = 80.0

RUN_SYNTHETIC_DEMO = True
RUN_WSI_BATCH = False
RUN_TMA_BATCH = False

## Synthetic demo

Two tissue blobs on a plain TIFF, extracted at 5x. This checks the whole
extraction path in seconds.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

demo_slides = DEMO_ROOT / "extraction" / "slides"
demo_slides.mkdir(parents=True, exist_ok=True)
canvas = Image.new("RGB", (1600, 1200), (245, 245, 245))
draw = ImageDraw.Draw(canvas)
draw.ellipse((150, 150, 750, 650), fill=(190, 110, 160))
draw.ellipse((950, 550, 1450, 1050), fill=(185, 105, 155))
canvas.save(demo_slides / "demo_slide.tif")

if RUN_SYNTHETIC_DEMO:
    demo = rp.extract_tissue(demo_slides, DEMO_ROOT / "extraction" / "regions",
                             source_magnification=20, target_magnification=5)
    for item in demo.by_role("region"):
        print(item.sample_id, item.path.name, item.meta["absolute_box"])

## Ordinary slides

Each slide writes `regions/tissue_extraction/<slide>/region_NNN.tif` with a
preview and a manifest. Regions whose files all exist are skipped
(`skip_existing=True`), so interrupted runs resume.

In [ ]:
tissue_settings = dict(
    target_magnification=TARGET_MAGNIFICATION,
    detection_magnification=1.25,
    source_magnification=WSI_SOURCE_MAGNIFICATION,
    min_area_fraction=0.005,
)

if RUN_WSI_BATCH:
    regions = rp.extract_tissue(WSI_INPUT_DIR, RESULTS_ROOT / "regions", **tissue_settings)
    print({slide: len(found) for slide, found in regions.summary["regions"].items()})
else:
    print("Set RUN_WSI_BATCH=True after checking WSI_INPUT_DIR.")

## TMA blocks

Slides are grouped into blocks by filename and recognized as H&E or a marker
by keyword. With `per_stain_detection=True`, cores are detected on every
stain and `fallback_to_he=True` reuses the H&E boxes when counts disagree.
Name custom markers in `stains`.

In [ ]:
tma_settings = dict(
    target_magnification=TARGET_MAGNIFICATION,
    source_magnification=TMA_SOURCE_MAGNIFICATION,
    min_area_fraction=0.0005,
    min_circularity=0.60,
    stains=["H&E", "CD8", "CD31"],
)

if RUN_TMA_BATCH:
    cores = rp.extract_tma(TMA_INPUT_DIR, RESULTS_ROOT / "cores", **tma_settings)
    print(cores.summary["slides"])
else:
    print("Set RUN_TMA_BATCH=True after checking TMA filenames.")

## Tuning guide

| Symptom | Setting |
|---|---|
| Small fragments missing | lower `min_area_fraction` |
| Dust becomes a region | raise `min_area_fraction` |
| Irregular cores rejected | lower `min_circularity` |
| Crop clips the core edge | raise `box_scale` slightly |
| IHC core detection fails | keep `ihc_enhance` and `fallback_to_he` on |
| Plain TIFF at the wrong scale | set `source_magnification` |

Tune on representative slides, record the settings, then apply one cohort
config. Continue with **03** for alignment.